# Task 2 – Finite Automata for Real-Time Validation
**Formal Language and Automata (21CSC301T)**

This notebook implements three deterministic finite automata (DFA) from the assignment:
1. Password verification
2. Simplified email-format validation
3. Four-digit ATM PIN verification

It runs accepted and rejected examples, displays the results, and reports the combined test summary. No external dataset is required; all test inputs are included below.

In [ ]:
# Cell 1: Environment setup
# This notebook uses Python's standard library and pandas for result tables.
# Install pandas only if it is not already available (Google Colab normally has it).
import sys, subprocess, importlib.util

if importlib.util.find_spec("pandas") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas"])

import re
import pandas as pd
from IPython.display import display, Markdown

print("Environment ready.")

## 1. DFA design

A DFA reads the input from left to right and changes state according to each character. If the input finishes in an accepting state, it is accepted; otherwise, it is rejected.

In [ ]:
# Cell 2: Password DFA
# Categories: U = uppercase, L = lowercase, D = digit, O = other non-whitespace.
# Whitespace goes to a trap state. Length is capped at 8.
def password_dfa(password):
    length = 0
    uppercase_seen = False
    lowercase_seen = False
    digit_seen = False

    for ch in password:
        if ch.isspace():
            return False
        if 'A' <= ch <= 'Z':
            uppercase_seen = True
        elif 'a' <= ch <= 'z':
            lowercase_seen = True
        elif '0' <= ch <= '9':
            digit_seen = True
        # Other non-whitespace characters are allowed as O.

        length = min(length + 1, 8)

    return length == 8 and uppercase_seen and lowercase_seen and digit_seen


password_tests = [
    ("Abcd1234", True), ("Hello123A", True), ("PassWord9", True),
    ("Student1A", True), ("College5B", True),
    ("hello123", False), ("ABCDEFG1", False), ("abcdefghi", False),
    ("Abc123", False), ("12345678", False), ("Abcd 1234", False)
]

password_results = []
for value, expected in password_tests:
    actual = password_dfa(value)
    password_results.append({
        "Input": value,
        "Result": "ACCEPTED" if actual else "REJECTED",
        "Expected": "ACCEPTED" if expected else "REJECTED",
        "Test": "PASS" if actual == expected else "FAIL"
    })

password_df = pd.DataFrame(password_results)
display(password_df)
print("Tracking states: 9 × 2 × 2 × 2 = 72")
print("Trap state: 1 | Total including trap: 73")
print("Initial state: (0, False, False, False)")
print("Accepting condition: length = 8, uppercase = True, lowercase = True, digit = True")

In [ ]:
# Cell 3: Email format DFA
# Simplified rule from the assignment:
# alphanumeric local part @ alphanumeric domain . alphabetic TLD (at least 2 chars)
def email_dfa(email):
    # Exactly one @; local part and domain must be non-empty.
    if email.count("@") != 1:
        return False

    local, domain_part = email.split("@")
    if not local or not domain_part:
        return False

    if not all(ch.isalnum() for ch in local):
        return False

    # Use the final dot to separate domain name and TLD.
    if "." not in domain_part:
        return False
    domain, tld = domain_part.rsplit(".", 1)

    if not domain or not tld:
        return False
    if not all(ch.isalnum() for ch in domain):
        return False
    if len(tld) < 2 or not all("A" <= ch <= "Z" or "a" <= ch <= "z" for ch in tld):
        return False

    return True


email_tests = [
    ("student@gmail.com", True), ("abc@college.edu", True),
    ("user12@yahoo.in", True), ("test1@domain.org", True),
    ("hello@school.net", True), ("studentgmail.com", False),
    ("@gmail.com", False), ("abc@.com", False),
    ("abc@college", False), ("abc@domain.c", False),
    ("a@@domain.com", False)
]

email_results = []
for value, expected in email_tests:
    actual = email_dfa(value)
    email_results.append({
        "Input": value,
        "Result": "ACCEPTED" if actual else "REJECTED",
        "Expected": "ACCEPTED" if expected else "REJECTED",
        "Test": "PASS" if actual == expected else "FAIL"
    })

email_df = pd.DataFrame(email_results)
display(email_df)

In [ ]:
# Cell 4: ATM PIN DFA
# q0 -> q1 -> q2 -> q3 -> q4 for exactly four digits.
# Any non-digit or any extra character goes to the dead state.
def atm_pin_dfa(pin):
    state = 0  # q0
    for ch in pin:
        if not ch.isdigit() or state == 4:
            return False
        state += 1
    return state == 4


pin_tests = [
    ("1234", True), ("5678", True), ("2468", True),
    ("1357", True), ("9087", True), ("123", False),
    ("12345", False), ("12A4", False), ("ABCD", False),
    ("1 23", False)
]

pin_results = []
for value, expected in pin_tests:
    actual = atm_pin_dfa(value)
    pin_results.append({
        "Input": value,
        "Result": "ACCEPTED" if actual else "REJECTED",
        "Expected": "ACCEPTED" if expected else "REJECTED",
        "Test": "PASS" if actual == expected else "FAIL"
    })

pin_df = pd.DataFrame(pin_results)
display(pin_df)
print("States: q0, q1, q2, q3, q4 and Dead")
print("Only q4 is accepting.")

## 2. Combined test runner

The following cell combines all 32 test cases (11 password + 11 email + 10 PIN) and checks each actual result against its expected result.

In [ ]:
# Cell 5: Combined verification
all_results = []

for row in password_results:
    all_results.append({"Application": "Password", **row})
for row in email_results:
    all_results.append({"Application": "Email", **row})
for row in pin_results:
    all_results.append({"Application": "ATM PIN", **row})

combined_df = pd.DataFrame(all_results)
display(combined_df)

total = len(combined_df)
passed = int((combined_df["Test"] == "PASS").sum())
failed = total - passed

print("=" * 48)
print(f"TOTAL TESTS: {total}")
print(f"PASSED: {passed}")
print(f"FAILED: {failed}")
print("Verification status:", "ALL TESTS PASSED" if failed == 0 else "SOME TESTS FAILED")

In [ ]:
# Cell 6: Visual overview of the three DFA applications
# A simple schematic overview (not a complete transition diagram).
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))

def draw_chain(ax, states, title, labels):
    ax.set_title(title, fontsize=11, weight="bold")
    ax.axis("off")
    xs = list(range(len(states)))
    for i, (x, state) in enumerate(zip(xs, states)):
        circle = Circle((x, 0), 0.25, fill=False, linewidth=2)
        ax.add_patch(circle)
        ax.text(x, 0, state, ha="center", va="center", fontsize=9)
        if i < len(states)-1:
            ax.annotate("", xy=(x+0.72, 0), xytext=(x+0.28, 0),
                        arrowprops=dict(arrowstyle="->", lw=1.4))
            ax.text(x+0.5, 0.18, labels[i], ha="center", fontsize=8)
    ax.set_xlim(-0.5, len(states)-0.5)
    ax.set_ylim(-0.6, 0.7)

draw_chain(axes[0], ["Start", "Track", "Accept"], "Password DFA",
           ["read chars", "all rules met"])
draw_chain(axes[1], ["START", "LOCAL", "DOMAIN", "TLD", "VALID"], "Email DFA",
           ["alnum", "@", "dot", "2+ letters"])
draw_chain(axes[2], ["q0", "q1", "q2", "q3", "q4"], "ATM PIN DFA",
           ["digit", "digit", "digit", "digit"])
plt.tight_layout()
plt.show()

## 3. Conclusion

The three DFAs validate input syntax:
- Password: at least 8 characters, with uppercase, lowercase, and digit.
- Email: simplified local-part, `@`, domain, and TLD structure.
- ATM PIN: exactly four decimal digits.

These checks validate format only. Password strength, whether an email mailbox exists, and matching a PIN against an account require additional application logic.